# Completed-run evaluation and comparison

Run on CPU, locally or in Colab. Reads saved predictions; no training or project imports.
Evaluates either model or both, verifies saved metrics and writes tables, PNG/PDF figures and
hash-checked error examples under each run's `evaluation/`. Set the explicit MobileViT run name
after notebook 02 completes to export paired same-image comparisons.

These are validation-selected development results. Parent-photo separation remains unverified.
Seed spread describes optimization variation and is not a confidence interval.


In [ ]:
import csv, hashlib, io, json, os, shutil, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve)

# Select explicit completed run names; no automatic newest-run selection.
CRACKFUSION_RUN_NAME = 'run_4d1bf234e5ee01bc'
MOBILEVIT_RUN_NAME = None  # Paste notebook 02's completed run name here.
EXAMPLE_SEED = 42
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Fridge/major_project_dataset')
else:
    BASE = next((p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / 'requirements.txt').is_file() and (p / 'notebooks').is_dir()), Path.cwd())
print('Results base:',BASE)


In [ ]:
def binary_metrics(labels, probabilities, threshold=0.5):
    labels, probabilities = np.asarray(labels).reshape(-1), np.asarray(probabilities).reshape(-1)
    predictions = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    both = len(np.unique(labels)) == 2
    return dict(accuracy=float(accuracy_score(labels, predictions)),
        balanced_accuracy=float(balanced_accuracy_score(labels, predictions)),
        precision=float(precision_score(labels, predictions, zero_division=0)),
        recall=float(recall_score(labels, predictions, zero_division=0)),
        specificity=float(tn / (tn + fp)) if tn + fp else None,
        f1=float(f1_score(labels, predictions, zero_division=0)),
        macro_f1=float(f1_score(labels, predictions, labels=[0, 1], average='macro', zero_division=0)),
        roc_auc=float(roc_auc_score(labels, probabilities)) if both else None,
        average_precision=float(average_precision_score(labels, probabilities)) if both else None,
        confusion_matrix=[[int(tn), int(fp)], [int(fn), int(tp)]])



In [ ]:
def evaluate_run(family, RUN_NAME):
    RUN = BASE / "outputs" / family / RUN_NAME
    OUT = RUN / "evaluation"
    if not (RUN / "completed_seeds.json").is_file():
        raise FileNotFoundError(f"Completed run not found: {RUN}. Finish notebook 03 or set RUN_NAME.")
    config = json.loads((RUN / "configuration.json").read_text())
    seeds = config["SEEDS"]
    assert seeds and len(seeds) == len(set(seeds))
    assert sorted(json.loads((RUN / "completed_seeds.json").read_text())) == sorted(seeds), "Some seeds are incomplete"
    assert config["PROTOCOL"] == "development", "This notebook evaluates the development validation protocol"
    assert hashlib.sha256((RUN / "manifest.csv").read_bytes()).hexdigest() == config["manifest_sha256"], "Manifest hash mismatch"
    threshold = float(config["THRESHOLD"])
    assert 0 <= threshold <= 1
    manifest = pd.read_csv(RUN / "manifest.csv", keep_default_na=False)
    assert not manifest.path.duplicated().any()
    validation = manifest[manifest.split == "dev_validation"].set_index("path")
    assert len(validation) and (validation.split_basis == "tile_development_only").all()
    OUT.mkdir(parents=True, exist_ok=True)
    print(f"Run: {RUN_NAME}; seeds: {seeds}; validation images per seed: {len(validation)}")
    print("Development validation only; parent-photo separation is unverified.")
    
    
    # Write each artifact completely before replacing its previous copy.
    def save_bytes(name, data):
        fd, temp = tempfile.mkstemp(prefix=".evaluation_", dir=OUT)
        try:
            with os.fdopen(fd, "wb") as f:
                f.write(data)
            os.replace(temp, OUT / name)
        finally:
            if os.path.exists(temp): os.unlink(temp)
    
    def save_report(name, data):
        save_bytes(name, (json.dumps(data, indent=2, allow_nan=False) + "\n").encode())
    
    def save_table(name, table):
        save_bytes(name, table.to_csv(index=False).encode())
    
    def save_figure(name, fig):
        fig.tight_layout()
        for extension in ("png", "pdf"):
            buffer = io.BytesIO()
            fig.savefig(buffer, format=extension, dpi=180, bbox_inches="tight")
            save_bytes(f"{name}.{extension}", buffer.getvalue())
        plt.show()
        plt.close(fig)
    
    def check_predictions(frame):
        assert len(frame) == len(validation) and not frame.path.duplicated().any(), "Missing or duplicate predictions"
        assert set(frame.path) == set(validation.index), "Predictions are not the validation set"
        aligned = validation.loc[frame.path]
        for field in ("label", "surface", "parent_id"):
            assert np.array_equal(frame[field].astype(str).to_numpy(), aligned[field].astype(str).to_numpy()), f"{field} mismatch"
        probabilities = frame.probability.to_numpy(dtype=float)
        assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all(), "Invalid probabilities"
        assert np.array_equal(frame.prediction.to_numpy(), (probabilities >= threshold).astype(int)), "Threshold mismatch"
    
    def compare_metrics(actual, saved):
        for key, value in actual.items():
            assert (value is None and saved[key] is None) or (value is not None and np.allclose(value, saved[key], atol=1e-8)), f"Metric mismatch: {key}"
    
    predictions, histories, metrics = {}, {}, {}
    rows, surfaces = [], []
    for seed in seeds:
        frame = pd.read_csv(RUN / f"predictions_seed_{seed}.csv", keep_default_na=False)
        check_predictions(frame)
        saved = json.loads((RUN / f"metrics_seed_{seed}.json").read_text())
        assert saved["seed"] == seed and saved["split"] == "validation" and saved["protocol"] == "development"
        metric = binary_metrics(frame.label, frame.probability, threshold)
        compare_metrics(metric, saved["metrics"])
        history = json.loads((RUN / f"history_seed_{seed}.json").read_text())
        assert history and [h["epoch"] for h in history] == list(range(1, len(history) + 1))
        best = max(history, key=lambda h: h["macro_f1"])
        assert np.isclose(best["macro_f1"], metric["macro_f1"]), "Selected checkpoint does not match history"
        rows.append(dict(seed=seed, selected_epoch=best["epoch"], stopping_epoch=history[-1]["epoch"],
                         **{k:v for k,v in metric.items() if k != "confusion_matrix"}))
        for surface, group in frame.groupby("surface"):
            values = binary_metrics(group.label, group.probability, threshold)
            compare_metrics(values, saved["per_surface"][surface])
            surfaces.append(dict(seed=seed, surface=surface, n=len(group), **{k:v for k,v in values.items() if k != "confusion_matrix"}))
        predictions[seed], histories[seed], metrics[seed] = frame, history, metric
        save_table(f"errors_seed_{seed}.csv", frame[frame.label != frame.prediction])
    per_seed, per_surface = pd.DataFrame(rows), pd.DataFrame(surfaces)
    metric_names = list(metrics[seeds[0]])[:-1]
    aggregate = {k:dict(mean=float(per_seed[k].mean()), seed_std=float(per_seed[k].std(ddof=1)) if len(seeds)>1 else None) for k in metric_names}
    for key, saved in json.loads((RUN / "summary.json").read_text()).items():
        compare_metrics(aggregate[key], saved)
    save_table("metrics_per_seed.csv", per_seed)
    save_table("metrics_per_surface.csv", per_surface)
    save_table("metrics_mean_seed_std.csv", pd.DataFrame(aggregate).T.rename_axis("metric").reset_index())
    display(per_seed.round(4))
    display(pd.DataFrame(aggregate).T.round(4))
    print("All saved overall, surface and aggregate metrics reproduced from predictions.")
    
    fig, axes = plt.subplots(len(seeds), 2, figsize=(11, 3 * len(seeds)), squeeze=False)
    for i, seed in enumerate(seeds):
        h = pd.DataFrame(histories[seed])
        selected = int(per_seed.set_index("seed").loc[seed, "selected_epoch"])
        axes[i,0].plot(h.epoch, h.train_loss, label="Train")
        axes[i,0].plot(h.epoch, h.validation_loss, label="Validation")
        axes[i,1].plot(h.epoch, h.macro_f1, label="Validation macro F1")
        for ax in axes[i]:
            ax.axvline(selected, color="gray", linestyle="--", label=f"Selected epoch {selected}")
            ax.set_xlabel("Epoch"); ax.set_title(f"Seed {seed}"); ax.legend(fontsize=8)
        axes[i,0].set_ylabel("BCE loss"); axes[i,1].set_ylabel("Macro F1"); axes[i,1].set_ylim(0,1)
    save_figure("learning_curves", fig)
    fig, axes = plt.subplots(1,len(seeds),figsize=(4*len(seeds),4),squeeze=False)
    for ax, seed in zip(axes[0], seeds):
        matrix = np.array(metrics[seed]["confusion_matrix"])
        ax.imshow(matrix, cmap="Blues", vmin=0, vmax=len(validation)/2)
        for (r,c), value in np.ndenumerate(matrix): ax.text(c,r,str(value),ha="center",va="center")
        ax.set(xticks=[0,1], yticks=[0,1], xticklabels=["Non-crack","Crack"], yticklabels=["Non-crack","Crack"],
               xlabel="Predicted", ylabel="Actual", title=f"Seed {seed}; threshold {threshold:g}")
    save_figure("confusion_matrices", fig)
    
    fig, axes = plt.subplots(1,2,figsize=(11,4))
    for seed in seeds:
        frame = predictions[seed]
        fpr,tpr,_ = roc_curve(frame.label, frame.probability)
        precision,recall,_ = precision_recall_curve(frame.label, frame.probability)
        axes[0].plot(fpr,tpr,label=f"Seed {seed}: AUC {metrics[seed]['roc_auc']:.3f}")
        axes[1].plot(recall,precision,label=f"Seed {seed}: AP {metrics[seed]['average_precision']:.3f}")
    axes[0].plot([0,1],[0,1],"k--",alpha=.4)
    axes[1].axhline(predictions[seeds[0]].label.mean(),color="gray",linestyle="--",label="Class prevalence")
    for ax in axes: ax.set(xlim=(0,1),ylim=(0,1)); ax.legend(fontsize=8)
    axes[0].set(xlabel="False positive rate",ylabel="True positive rate",title="Validation ROC")
    axes[1].set(xlabel="Recall",ylabel="Precision",title="Validation precision–recall")
    save_figure("roc_precision_recall", fig)
    fig, axes = plt.subplots(1,2,figsize=(10,4))
    for ax,key in zip(axes,["macro_f1","recall"]):
        grouped = per_surface.groupby("surface")[key]
        means, spread = grouped.mean(), grouped.std(ddof=1).fillna(0)
        ax.bar(means.index,means,yerr=spread,capsize=5,color=["#527fa3","#759b78","#bd9963"])
        ax.set(ylim=(0,1),ylabel=key.replace("_"," "),title="Development validation by surface")
    save_figure("surface_performance", fig)
    
    assert EXAMPLE_SEED in seeds
    bundle = next((p for p in (BASE / "crackfusion_colab_bundle.zip", BASE / "outputs/data_audit/crackfusion_colab_bundle.zip") if p.is_file()), None)
    assert bundle is not None, "Place crackfusion_colab_bundle.zip in the dataset folder or outputs/data_audit"
    # Copy the small bundle once to local scratch to avoid repeated Drive reads.
    with tempfile.TemporaryDirectory() as scratch:
        local_bundle = Path(scratch) / "bundle.zip"
        shutil.copyfile(bundle, local_bundle)
        with zipfile.ZipFile(local_bundle) as archive:
            frame = predictions[EXAMPLE_SEED]
            groups = {
                "false_positives": frame[(frame.label==0)&(frame.prediction==1)].sort_values("probability",ascending=False),
                "false_negatives": frame[(frame.label==1)&(frame.prediction==0)].sort_values("probability"),
                "correct_examples": frame[frame.label==frame.prediction],
            }
            for name, group in groups.items():
                fig, axes = plt.subplots(2,4,figsize=(12,6))
                for ax in axes.flat: ax.axis("off")
                for ax, (_,row) in zip(axes.flat,group.head(8).iterrows()):
                    relative = PurePosixPath(row.path)
                    assert not relative.is_absolute() and ".." not in relative.parts
                    data = archive.read("dataset/SDNET2018/" + row.path)
                    assert hashlib.sha256(data).hexdigest() == validation.loc[row.path,"sha256"], f"Image hash mismatch: {row.path}"
                    with Image.open(io.BytesIO(data)) as image: ax.imshow(image.convert("RGB"))
                    ax.set_title(f"{row.path}\n{row.surface}; true={row.label}, pred={row.prediction}\nP(crack)={row.probability:.3f}",fontsize=8)
                fig.suptitle(f"{name.replace('_',' ').title()} — seed {EXAMPLE_SEED}; threshold {threshold:g}")
                save_figure(name,fig)
    report = dict(run=RUN_NAME, seeds=seeds, validation_images_per_seed=len(validation), threshold=threshold,
        protocol="development", split_basis="tile_development_only", manifest_sha256=config["manifest_sha256"],
        example_seed=EXAMPLE_SEED, aggregate=aggregate, selected_epochs=per_seed[["seed","selected_epoch","stopping_epoch"]].to_dict("records"),
        limitation="Validation-selected development results; parent-photo separation unverified. Seed spread is not a confidence interval.",
        artifacts=sorted(p.name for p in OUT.iterdir() if p.is_file() and not p.name.startswith(".") and p.name != "evaluation_summary.json"))
    save_report("evaluation_summary.json",report)
    print(f"Evaluation complete. Tables, PNG/PDF figures, error lists and summary saved to {OUT}")
    
    return dict(run=RUN,config=config,seeds=seeds,manifest=validation,predictions=predictions,per_seed=per_seed,aggregate=aggregate)


In [ ]:
reports = {}
for family,name in [('crackfusion_tiny',CRACKFUSION_RUN_NAME),('mobilevit_baseline',MOBILEVIT_RUN_NAME)]:
    if name:
        reports[family] = evaluate_run(family,name)
if len(reports) == 2:
    cf,mv = reports['crackfusion_tiny'],reports['mobilevit_baseline']
    assert cf['config']['manifest_sha256'] == mv['config']['manifest_sha256'], 'Different audited manifests'
    assert set(cf['seeds']) == set(mv['seeds']), 'Different seed sets'
    out = BASE / 'outputs/comparisons' / (cf['run'].name+'__'+mv['run'].name)
    out.mkdir(parents=True,exist_ok=True)
    differences,paired = [],[]
    for seed in cf['seeds']:
        left = cf['predictions'][seed].set_index('path').sort_index()
        right = mv['predictions'][seed].set_index('path').sort_index()
        assert left.index.equals(right.index)
        assert left[['label','surface','parent_id']].equals(right[['label','surface','parent_id']])
        row = {'seed':seed}
        for key in ('macro_f1','f1','recall','balanced_accuracy','average_precision'):
            row[key+'_crackfusion_minus_mobilevit'] = cf['per_seed'].set_index('seed').loc[seed,key]-mv['per_seed'].set_index('seed').loc[seed,key]
        differences.append(row)
        a,b = left.prediction == left.label, right.prediction == right.label
        paired.append(dict(seed=seed,n=len(left),both_correct=int((a&b).sum()),
            only_crackfusion_correct=int((a&~b).sum()),only_mobilevit_correct=int((~a&b).sum()),both_wrong=int((~a&~b).sum())))
    for name,rows in [('metric_differences.csv',differences),('paired_prediction_errors.csv',paired)]:
        path=out/name; temporary=path.with_suffix('.csv.tmp')
        pd.DataFrame(rows).to_csv(temporary,index=False); temporary.replace(path)
    display(pd.DataFrame(differences))
    print('Paired development comparison saved:',out)
    print('Pretrained reference with different size/preprocessing/budget; seed differences are descriptive, not parent-cluster confidence intervals.')
else:
    print('Set both run names after MobileViT finishes to export paired comparisons.')


## Optional MobileViT architecture inspection
Retains the observed feature-flow, block anatomy, module execution and parameter reports.


In [ ]:
INSPECT_MOBILEVIT = False  # Requires a completed MobileViT run and PyTorch/timm.
if INSPECT_MOBILEVIT:
    import torch, timm
    from timm.data import create_transform
    from matplotlib.patches import FancyBboxPatch
    assert 'mobilevit_baseline' in reports, 'Select MOBILEVIT_RUN_NAME first'
    report = reports['mobilevit_baseline']
    OUTPUT_DIR = report['run'] / 'evaluation'
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(report['run'] / f'best_seed_{EXAMPLE_SEED}.pt',map_location=DEVICE,weights_only=True)
    model = timm.create_model(checkpoint['model_name'],pretrained=False,num_classes=1).to(DEVICE).eval()
    model.load_state_dict(checkpoint['model'])
    validation_transform = create_transform(**checkpoint['data_config'],is_training=False)
    validation_rows = report['manifest'].reset_index().to_dict('records')
    with tempfile.TemporaryDirectory() as scratch:
        dataset_root = Path(scratch) / 'dataset/SDNET2018'
        bundle = next(p for p in [BASE/'crackfusion_colab_bundle.zip',BASE/'outputs/data_audit/crackfusion_colab_bundle.zip'] if p.is_file())
        with zipfile.ZipFile(bundle) as archive:
            sample = validation_rows[0]['path']
            relative = PurePosixPath(sample)
            assert not relative.is_absolute() and '..' not in relative.parts
            image_data = archive.read('dataset/SDNET2018/'+sample)
            assert hashlib.sha256(image_data).hexdigest() == validation_rows[0]['sha256']
            target=dataset_root/sample; target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(image_data)
        feature_model = timm.create_model(checkpoint["model_name"], pretrained=False, features_only=True).to(DEVICE).eval()
        dummy = torch.zeros(1, *checkpoint["data_config"]["input_size"], device=DEVICE)
        with torch.inference_mode():
            feature_shapes = [list(feature.shape) for feature in feature_model(dummy)]
        del feature_model, dummy
        
        flow_labels = ["Input\n3×256×256"] + [
            f"Stage {index}\n{shape[1]}×{shape[2]}×{shape[3]}"
            for index, shape in enumerate(feature_shapes, start=1)
        ] + ["Global pool\n+ 1 logit"]
        colors = ["#E8F1FB"] + ["#DDF3E4"] * len(feature_shapes) + ["#FBE8D5"]
        figure, axis = plt.subplots(figsize=(14, 3.2))
        axis.set_xlim(0, len(flow_labels) * 2.1)
        axis.set_ylim(0, 3)
        axis.axis("off")
        for index, (label, color) in enumerate(zip(flow_labels, colors)):
            x = index * 2.1 + 0.2
            box = FancyBboxPatch(
                (x, 0.9), 1.65, 1.15, boxstyle="round,pad=0.08",
                facecolor=color, edgecolor="#34495E", linewidth=1.3,
            )
            axis.add_patch(box)
            axis.text(x + 0.825, 1.475, label, ha="center", va="center", fontsize=10)
            if index < len(flow_labels) - 1:
                axis.annotate("", xy=(x + 2.02, 1.475), xytext=(x + 1.67, 1.475), arrowprops={"arrowstyle": "->"})
        axis.set_title("MobileViT-XS feature flow (observed tensor shapes)", fontsize=14)
        figure.tight_layout()
        flow_path = OUTPUT_DIR / "mobilevit_feature_flow.png"
        figure.savefig(flow_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Feature shapes:", feature_shapes)
        print("Saved", flow_path.resolve())
        
        sample_row = validation_rows[0]
        with Image.open(dataset_root / sample_row["path"]) as image:
            batch = validation_transform(image.convert("RGB")).unsqueeze(0).to(DEVICE)
        
        mobilevit_blocks = [
            (name, module)
            for name, module in model.named_modules()
            if "mobilevitblock" in module.__class__.__name__.lower()
        ]
        if not mobilevit_blocks:
            raise RuntimeError("No MobileViTBlock modules were found in the loaded model.")
        
        total_parameters = sum(parameter.numel() for parameter in model.parameters())
        print(f"Model: {model.__class__.__name__} | parameters: {total_parameters:,}")
        for name, block in mobilevit_blocks:
            parameters = sum(parameter.numel() for parameter in block.parameters())
            children = [f"{child_name}:{child.__class__.__name__}" for child_name, child in block.named_children()]
            print(f"{name} | {block.__class__.__name__} | {parameters:,} parameters")
            print("  " + " -> ".join(children))
        
        def tensor_shapes(value):
            if isinstance(value, torch.Tensor):
                return [list(value.shape)]
            if isinstance(value, (tuple, list)):
                shapes = []
                for item in value:
                    shapes.extend(tensor_shapes(item))
                return shapes
            return []
        
        block_records = {}
        component_execution = []
        handles = []
        
        def block_hook(name):
            def capture(module, inputs, output):
                block_records[name] = {
                    "type": module.__class__.__name__,
                    "input_shapes": tensor_shapes(inputs),
                    "output_shapes": tensor_shapes(output),
                    "parameters": sum(parameter.numel() for parameter in module.parameters()),
                }
            return capture
        
        def component_hook(block_name, component_name):
            def capture(module, inputs, output):
                component_execution.append({
                    "block": block_name,
                    "component": component_name,
                    "type": module.__class__.__name__,
                    "input_shapes": tensor_shapes(inputs),
                    "output_shapes": tensor_shapes(output),
                    "parameters": sum(parameter.numel() for parameter in module.parameters()),
                })
            return capture
        
        for block_name, block in mobilevit_blocks:
            handles.append(block.register_forward_hook(block_hook(block_name)))
            for component_name, component in block.named_children():
                handles.append(component.register_forward_hook(component_hook(block_name, component_name)))
        
        with torch.inference_mode():
            logit = model(batch)
        for handle in handles:
            handle.remove()
        
        assert len(block_records) == len(mobilevit_blocks)
        print("Output logit:", logit.item())
        for name, record in block_records.items():
            print(f"{name}: {record['input_shapes']} -> {record['output_shapes']} | {record['parameters']:,} parameters")
        print("Observed direct-component executions:", len(component_execution))
        
        stage_parameter_rows = [
            {
                "stage": name,
                "type": module.__class__.__name__,
                "parameters": sum(parameter.numel() for parameter in module.parameters()),
            }
            for name, module in model.named_children()
        ]
        assert sum(row["parameters"] for row in stage_parameter_rows) == total_parameters
        
        with (OUTPUT_DIR / "stage_parameters.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=["stage", "type", "parameters"])
            writer.writeheader()
            writer.writerows(stage_parameter_rows)
        
        with (OUTPUT_DIR / "mobilevit_components.csv").open("w", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(
                stream,
                fieldnames=["block", "component", "type", "input_shapes", "output_shapes", "parameters"],
            )
            writer.writeheader()
            writer.writerows(component_execution)
        
        print(f"{'Stage':<20}{'Type':<24}{'Parameters':>14}")
        for row in stage_parameter_rows:
            print(f"{row['stage']:<20}{row['type']:<24}{row['parameters']:>14,}")
        
        first_name = next(iter(block_records))
        first_record = block_records[first_name]
        input_shape = first_record["input_shapes"][0]
        output_shape = first_record["output_shapes"][0]
        steps = [
            (f"Input map\n{input_shape[1]}×{input_shape[2]}×{input_shape[3]}", "#E8F1FB"),
            ("Local convolution\nspatial texture", "#DDF3E4"),
            ("Unfold patches\nfeature tokens", "#FFF2CC"),
            ("Transformer\nglobal context", "#EADCF8"),
            ("Fold to map\nspatial alignment", "#FFF2CC"),
            ("Concat + convolution\nlocal/global fusion", "#FADBD8"),
            (f"Output map\n{output_shape[1]}×{output_shape[2]}×{output_shape[3]}", "#E8F1FB"),
        ]
        
        figure, axis = plt.subplots(figsize=(15, 3.2))
        axis.set_xlim(0, len(steps) * 2.1)
        axis.set_ylim(0, 3)
        axis.axis("off")
        for index, (label, color) in enumerate(steps):
            x = index * 2.1 + 0.2
            axis.add_patch(FancyBboxPatch(
                (x, 0.9), 1.65, 1.15, boxstyle="round,pad=0.08",
                facecolor=color, edgecolor="#34495E", linewidth=1.3,
            ))
            axis.text(x + 0.825, 1.475, label, ha="center", va="center", fontsize=9.5)
            if index < len(steps) - 1:
                axis.annotate("", xy=(x + 2.02, 1.475), xytext=(x + 1.67, 1.475), arrowprops={"arrowstyle": "->"})
        axis.set_title(f"{first_name}: MobileViT local/global processing and fusion", fontsize=14)
        figure.tight_layout()
        diagram_path = OUTPUT_DIR / "mobilevit_block_anatomy.png"
        figure.savefig(diagram_path, dpi=180, bbox_inches="tight")
        plt.show()
        print("Saved", diagram_path.resolve())
        
        report = {
            "model_name": checkpoint["model_name"],
            "total_parameters": total_parameters,
            "sample_path": sample_row["path"],
            "input_shape": list(batch.shape),
            "output_shape": list(logit.shape),
            "mobilevit_block_count": len(mobilevit_blocks),
            "mobilevit_blocks": block_records,
            "top_level_stage_parameters": stage_parameter_rows,
            "component_execution": component_execution,
            "diagram": str(diagram_path.resolve()),
            "interpretation": (
                "Convolution preserves local crack texture; patch-wise Transformer processing adds global context; "
                "the folded global representation is fused back with spatial features."
            ),
        }
        report_path = OUTPUT_DIR / "architecture_report.json"
        report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
        
        for path in [report_path, diagram_path, OUTPUT_DIR / "stage_parameters.csv", OUTPUT_DIR / "mobilevit_components.csv"]:
            assert path.exists(), path
        print(json.dumps({
            "mobilevit_blocks": len(mobilevit_blocks),
            "block_names": list(block_records),
            "total_parameters": total_parameters,
            "report": str(report_path.resolve()),
        }, indent=2))
